# Analysis and Grouping for Bayesian Optimization

Please get in contact with Carla if you want to create an overview table of your data with certain parameters.

In [1]:
author_names_list = ['Sanam Hosseinzadeh shakib', 'Aaditya Vashisht', 'Maitryi Gupta']

# TODO calculation of targets

PARAMETER_MAPPING = {
    'Sanam Hosseinzadeh shakib': {
        'upload_id': '',
        'sample_id': 'ca_data.samples[0].name',
        'potential1 (mV vs Hg/HgO)': 'ca_data.properties.step_1_potential',   #E1
        'hold1 (s)': 'ca_data.properties.step_1_time',                        #hold1
        'potential2 (mV vs Hg/HgO)': 'cv_data.properties.limit_potential_1',  #E2
        'potential3 (mV vs Hg/HgO)': 'cv_data.properties.limit_potential_2',  #E3
        'environment_id': 'ca_data.environment.lab_id',
        'K2CO3':      'ca_data.environment.substances[0].concentration_mmol_per_l',
        'cycles':     'cv_data.properties.cycles',
        #'ca_voltage_shift': 'ca_data.voltage_shift',   # needed if we want to get potential vs RHE
        #'cv_voltage_shift': 'cv_data.voltage_shift',   # needed if we want to get potential vs RHE
    },
    'Aaditya Vashisht': {
        'upload_id': '',
        'sample_id': 'ca_data.samples[0].name',
        'potential1 (mV vs Hg/HgO)': 'ca_data.properties.step_1_potential',   #E1
        'hold1 (s)': 'ca_data.properties.step_1_time',                        #hold1
        'potential2 (mV vs Hg/HgO)': 'cv_data.properties.limit_potential_1',  #E2
        'potential3 (mV vs Hg/HgO)': 'cv_data.properties.limit_potential_2',  #E3
        'cycles':     'cv_data.properties.cycles',
        #'ca_voltage_shift': 'ca_data.voltage_shift',   # needed if we want to get potential vs RHE
        #'cv_voltage_shift': 'cv_data.voltage_shift',   # needed if we want to get potential vs RHE
    },
    'Maitryi Gupta': {
        'upload_id': '',
        'sample_id': 'ca_data.samples[0].name',
        'potential1 (mV vs Hg/HgO)': 'ca_data.properties.step_1_potential',
        'hold1 (s)': 'ca_data.properties.step_1_time',
        'potential2 (mV vs Hg/HgO)': 'cv_data.properties.limit_potential_1',
        'potential3 (mV vs Hg/HgO)': 'cv_data.properties.limit_potential_2',
        'sweep speed (mV/s)': 'cv_data.properties.scan_rate',
        'cycle (P2-P3)': 'cv_data.properties.cycles',  
    },
    'other user': {
        'upload_id':  'id',
        'sample_id':  'ca_data.samples[0].name',
    }
}

CALCULATIONS_MAPPING = {
    'Sanam Hosseinzadeh shakib': {
        'potential1 (mV vs Hg/HgO)': lambda df: df['potential1 (mV vs Hg/HgO)'] * 1000, #+ df['ca_voltage_shift'],
        'potential2 (mV vs Hg/HgO)': lambda df: df['potential2 (mV vs Hg/HgO)'] * 1000, #+ df['cv_voltage_shift'],
        'potential3 (mV vs Hg/HgO)': lambda df: df['potential3 (mV vs Hg/HgO)'] * 1000, #+ df['cv_voltage_shift'],        
        'K2CO3': lambda df: df['K2CO3'] / 1000, # mol/l 
    },
    'Aaditya Vashisht': {
        'potential1 (mV vs Hg/HgO)': lambda df: (
                df.get('potential1 (mV vs Hg/HgO)').mul(1000) #+ df['ca_voltage_shift'],
                if df.get('potential1 (mV vs Hg/HgO)') is not None
                else pd.Series(None, index=df.index)
            ),
        'potential2 (mV vs Hg/HgO)': lambda df: (
                df.get('potential2 (mV vs Hg/HgO)').mul(1000) #+ df['cv_voltage_shift'],
                if df.get('potential2 (mV vs Hg/HgO)') is not None
                else pd.Series(None, index=df.index)
            ),
        'potential3 (mV vs Hg/HgO)': lambda df: (
                df.get('potential3 (mV vs Hg/HgO)').mul(1000) #+ df['cv_voltage_shift'],
                if df.get('potential3 (mV vs Hg/HgO)') is not None
                else pd.Series(None, index=df.index)
            ),
    },
    'Maitryi Gupta': {
        'hold2 (s)': lambda df: 0,
        'potential1 (mV vs Hg/HgO)': lambda df: df['potential1 (mV vs Hg/HgO)'] * 1000, # + df['ca_voltage_shift'],
        'potential2 (mV vs Hg/HgO)': lambda df: df['potential2 (mV vs Hg/HgO)'] * 1000, # + df['cv_voltage_shift'],
        'potential3 (mV vs Hg/HgO)': lambda df: df['potential3 (mV vs Hg/HgO)'] * 1000, # + df['cv_voltage_shift'],
        'interval1': lambda df: df['hold1 (s)'] + np.abs(df['potential1 (mV vs Hg/HgO)'] - df['potential2 (mV vs Hg/HgO)'])/df['sweep speed (mV/s)'], 
        'interval2': lambda df: df['hold2 (s)'] + np.abs(df['potential2 (mV vs Hg/HgO)'] - df['potential3 (mV vs Hg/HgO)'])/df['sweep speed (mV/s)'],
        'duration (s)': lambda df: df['interval1'] + df['cycle (P2-P3)'] * df['interval2'],
        'duration (h)': lambda df: df['duration (s)'] / 3600,
    },
    'other_user': {
    },
}

GROUP_REPETITIONS_MAPPING = {
    'Sanam Hosseinzadeh shakib': [
        'potential1 (mV vs Hg/HgO)',
        'hold1 (s)',
        'potential3 (mV vs Hg/HgO)',
        'K2CO3',
    ],
    'Aaditya Vashisht': [
        'potential1 (mV vs Hg/HgO)',
        'hold1 (s)',
        'potential3 (mV vs Hg/HgO)',
    ],
    'Maitryi Gupta': [
        'potential1 (mV vs Hg/HgO)',
        'hold1 (s)',
        'potential3 (mV vs Hg/HgO)',
        'sweep speed (mV/s)',
        'cycle (P2-P3)'
    ]
}

BAYBE_COL_NAMES_EXPORT_MAPPING = {
    'Sanam Hosseinzadeh shakib': ['cp_geom_mean', 'potential1 (mV vs Hg/HgO)', 'hold1 (s)', 'potential3 (mV vs Hg/HgO)', 'K2CO3',],
    'Aaditya Vashis': ['cp_geom_mean', 'potential1 (mV vs Hg/HgO)', 'hold1 (s)', 'potential3 (mV vs Hg/HgO)',],
    'Maitryi Gupta': ['cp_geom_mean', 'potential1 (mV vs Hg/HgO)', 'hold1 (s)', 'potential3 (mV vs Hg/HgO)', 'sweep speed (mV/s)', 'cycle (P2-P3)',]
}

In [2]:
%%capture
%matplotlib widget
#!pip install requests_cache

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import time
import requests
import pandas as pd
import numpy as np
import datetime
import math
import re

import ipywidgets as widgets
from IPython.display import display, clear_output

import sys
sys.path.insert(1, '../python-scripts-c6fxKDJrSsWp1xCxON1Y7g')
sys.path.insert(1, '../../python-scripts-c6fxKDJrSsWp1xCxON1Y7g')
from api_calls import *

#url = "https://nomad-hzb-ce.de/nomad-oasis/api/v1"
url = "https://nomad-hzb-ce.helmholtz-berlin.de/nomad-oasis/api/v1"

import os
os.environ["HTTP_PROXY"]="http://proxy.csn29.bessy.de:3128"
os.environ["HTTPS_PROXY"]="http://proxy.csn29.bessy.de:3128"
token = os.environ['NOMAD_CLIENT_ACCESS_TOKEN']

In [3]:
def get_upload_ids_from_main_authors(url, token, main_authors):   
    query = {
        'required': {
            'upload_id': '*',
        },
        'owner': 'visible',
        'query': {
            'origin:any': main_authors,
        },
        'pagination': {
            'page_size': 1000
        }
    }
    response = requests.post(f'{url}/entries/archive/query',
                             headers={'Authorization': f'Bearer {token}'}, json=query)
    linked_data = response.json()["data"]
    res = set()
    for ldata in linked_data:
        res.add(ldata.get('upload_id'))
    return res

def get_specific_entrytype_of_upload_ids(url, token, upload_list, entry_type):   
    query = {
        'required': {
            'data': '*',
        },
        'owner': 'visible',
        'query': {
            'upload_id:any': upload_list,
            'entry_type': entry_type
        },
        'pagination': {
            'page_size': 10000
        }
    }
    response = requests.post(f'{url}/entries/archive/query',
                             headers={'Authorization': f'Bearer {token}'}, json=query)
    linked_data = response.json()['data']
    res = []
    for ldata in linked_data:
        res.append(ldata['archive']['data'])
    return res 

def get_upload_name_from_id(url, token, upload_id):
    response = requests.get(f'{url}/uploads/{upload_id}', headers={'Authorization': f'Bearer {token}'})
    linked_data = response.json()['data']
    return linked_data.get('upload_name')


from nomad.client.archive import ArchiveQuery
def get_specific_entrytype_of_upload_ids_archivequery(url, token, upload_list, entry_type):   
    query = {
            'upload_id:any': upload_list,
            'entry_type': entry_type,
    }
    required = {'data': '*'}

    q = ArchiveQuery(query=query, required=required, owner='visible', page_size=1000, url=url)
    linked_data = q.download()
    res = []
    for entry_archive in linked_data:
        res.append(entry_archive.get('data'))
    if len(res) == 0:
        return [None]
    res_sorted = sorted(res, key=lambda x: x.get('datetime'), reverse=True)
    return res_sorted 

WARNING  MDAnalysis.coordinat 2026-07-27T09:19:30Z netCDF4 is not available. Writing AMBER ncdf files will be slow.
  - nomad.commit: 
  - nomad.deployment: oasis
  - nomad.service: unknown nomad service
  - nomad.version: 1.4.2
  - taskName: Task-52


In [4]:
# all ipywidgets

author_selector = widgets.Dropdown(
    options=author_names_list,
    value=author_names_list[0],
    description='NOMAD author:',
    style={'description_width': 'initial'}
)

cp_pre_step_current_selector = widgets.Dropdown(
    options=[None, -10, -500, 500],
    value=-500,
    description='CP pre step current (mA):',
    style={'description_width': 'initial'}
)

group_selector = widgets.Dropdown(
    description='Select parameter set:',
    style={'description_width': 'initial'}
)

get_button = widgets.Button(
    description='Get NOMAD data',
    button_style='success',
    layout=widgets.Layout(width='auto')
)

analysis_button = widgets.Button(
    description='Group and evaluate data',
    button_style='info',
    layout=widgets.Layout(width='auto')
)


save_baybe_button = widgets.Button(
    description='Save data for Bayesian Optimization',
    button_style='primary',
    layout=widgets.Layout(width='auto')
)

show_group_details_button = widgets.Button(
    description='Show plots for selected parameter set',
    button_style='info',
    layout=widgets.Layout(width='auto')
)

all_runs_output = widgets.Output()
analysis_output = widgets.Output()
baybe_output = widgets.Output()
group_select_output = widgets.Output()
group_detail_output = widgets.Output()

### Select Uploads

In [5]:
def on_author_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        all_runs_output.clear_output()
        analysis_output.clear_output()
        baybe_output.clear_output()
        group_select_output.clear_output()
        group_detail_output.clear_output()

author_selector.observe(on_author_change)
cp_pre_step_current_selector.observe(on_author_change)

display(author_selector, cp_pre_step_current_selector)

Dropdown(description='NOMAD author:', options=('Sanam Hosseinzadeh shakib', 'Aaditya Vashisht', 'Maitryi Gupta…

Dropdown(description='CP pre step current (mA):', index=2, options=(None, -10, -500, 500), style=DescriptionSt…

### Create Table from NOMAD entries

In [6]:
def get_nested_json_value(obj, path, default=None):
    parts = re.split(r'\.(?![^\[]*\])', path)
    for part in parts:
        if obj is None:
            return default
        m = re.match(r'^(\w+)\[(\d+)\]$', part)
        if m:
            key, idx = m.group(1), int(m.group(2))
            try:
                obj = obj.get(key)          # no default argument for NOMAD MSection objects
            except TypeError:
                obj = getattr(obj, key, None)  # this is the fallback for MSection
            obj = obj[idx] if isinstance(obj, list) and idx < len(obj) else None
        else:
            try:
                obj = obj.get(part)
            except TypeError:
                obj = getattr(obj, part, None)  # access attribut as fallback
    return obj if obj is not None else default

def apply_transforms(df, transform_map):
    df = df.copy()
    for col, fn in transform_map.items():
        df[col] = fn(df)
    return df

def get_result_df_from_upload_ids(upload_ids):
    rows = []
    for upload_id in upload_ids:
        try:
            data = {
                'ca_data': get_specific_entrytype_of_upload_ids_archivequery(url, token, [upload_id], 'CE_NOME_Chronoamperometry',)[0],
                'cv_data': get_specific_entrytype_of_upload_ids_archivequery(url, token, [upload_id], 'CE_NOME_CyclicVoltammetry',)[0],
            }
            #row_dict = {col: get_nested_json_value(data, path, default=0) for col, path in PARAMETER_MAPPING.get(author_selector.value).items()}
            row_dict = {col: get_nested_json_value(data, path, default=None) for col, path in PARAMETER_MAPPING.get(author_selector.value).items()}
            row_dict['upload_id'] = upload_id
            rows.append(row_dict)
        except:
            print(upload_id, "didnt work - maybe calibration upload?")
    result = pd.DataFrame(rows)
    #result = result.sort_values(by=['sample_id']).reset_index(drop = True)
    #result = result.sort_values(by=['sample_id'], key=lambda col: pd.to_numeric(col, errors='coerce')).reset_index(drop=True)
    result = result.sort_values(by=['sample_id'], key=lambda col: col.astype(str)).reset_index(drop=True)
    result = apply_transforms(result, CALCULATIONS_MAPPING.get(author_selector.value, pd.DataFrame()))
    return result

In [7]:
def on_button_clicked(b):
    global result
    with all_runs_output:
        all_runs_output.clear_output()
        print('Getting data. This can take some time...')
        author_list = [author_selector.value] #.split("-", 1)[-1]]
        upload_ids = get_upload_ids_from_main_authors(url, token, author_list)
        result = get_result_df_from_upload_ids(upload_ids)
        all_runs_output.clear_output()
        display(result)

get_button.on_click(on_button_clicked)

display(get_button, all_runs_output)

Button(button_style='success', description='Get NOMAD data', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Calculation of Targets

In [8]:
# TODO: should this be done on RHE compensated data? E_shift referenzelektrode
# TODO decide how table and activity is connected: everything in the same upload or match via sample ids?

In [9]:
def get_groups(df):
    res_grouped = df.groupby(
        GROUP_REPETITIONS_MAPPING.get(author_selector.value, [])
    ).agg({
        'upload_id': lambda x: list(x.unique()),
        'sample_id': lambda x: list(x.unique())
    }).reset_index()
    res_grouped.rename(columns={
        'upload_id': 'upload_ids',
        'sample_id': 'sample_ids'
    }, inplace=True)

    # sort by date
    res_grouped['sort_by_id_date'] = res_grouped['sample_ids'].apply(lambda x: str(x[0]) if x else '') #apply(lambda x: x[0][13:24] if x else '')
    res_grouped.sort_values(by='sort_by_id_date', inplace=True)
    res_grouped.drop(columns='sort_by_id_date', inplace=True)
    res_grouped.reset_index(drop=True, inplace=True)

    return res_grouped

def get_mean_std_no_cycles(data_list, quantity):
    replicates = []
    for measurement in data_list:
        if cp_pre_step_current_selector.value is not None and measurement.get('properties', {}).get('pre_step_current') != cp_pre_step_current_selector.value / 1000:
            #print(f'Skipped measurement "{measurement.get('name')}", {measurement.get('datetime')}, {measurement.get('samples', [{}])[0].get('name')} because of different pre_step_current.')
            #print(f'({cp_pre_step_current_selector.value / 1000} A != {measurement.get('properties', {}).get('pre_step_current')} A)')
            continue
        #if measurement.get('method') != 'OER Chronopotentiometry':
        if len(measurement.get(quantity)) != 301:
            print(f'Skipped measurement "{measurement.get('name')}", {measurement.get('datetime')}, {measurement.get('samples', [{}])[0].get('name')}  because of different {quantity} length.')
            continue
        replicates.append(measurement.get(quantity))
    mean_all = np.mean(replicates, axis=0)
    std_all = np.std(replicates, axis=0, ddof=1)
    mean_val = mean_all.mean() * 1000 #mV
    std_val = std_all.mean() * 1000 #mV
    return mean_val, std_val

def get_mean_std_of_groups(res_grouped):
    eval_col_names = ['cp_voltage_mean', 'cp_voltage_std', 'cp_geom_mean', 'cp_voltage_rhe_mean', 'cp_voltage_rhe_std', 'cp_geom_mean_rhe',
                      'cp_voltage_shifted_std', 'cp_geom_mean_voltage_shifted','cp_voltage_resistance_corrected_std', 'cp_geom_mean_resistance_corrected']
    eval_cols = []
    
    for group in res_grouped.itertuples():
        cp_data = get_specific_entrytype_of_upload_ids(url, token, group.upload_ids, 'CE_NOME_Chronopotentiometry',)
        
        cp_mean, cp_std = get_mean_std_no_cycles(cp_data, 'voltage')
        cp_mean_rhe, cp_std_rhe = get_mean_std_no_cycles(cp_data, 'voltage_rhe_compensated')
        cp_mean_rhe_u, cp_std_rhe_u = get_mean_std_no_cycles(cp_data, 'voltage_rhe_uncompensated')
        cp_mean_ref, cp_std_ref = get_mean_std_no_cycles(cp_data, 'voltage_ref_compensated')

        geom_mean = (cp_mean*cp_mean*cp_std)**(1/3)
        geom_mean_rhe = (cp_mean_rhe*cp_mean_rhe*cp_std_rhe)**(1/3)
        
        eval_cols.append([cp_mean, cp_std, geom_mean, cp_mean_rhe, cp_std_rhe, geom_mean_rhe, cp_mean_rhe_u, cp_std_rhe_u, cp_mean_ref, cp_std_ref])
    
    res_grouped.loc[:, eval_col_names] = eval_cols
    return res_grouped
    
#upload_ids = get_upload_ids_from_main_authors(url, token, [author_selector.value])
#result = get_result_df_from_upload_ids(upload_ids)
#res_grouped = get_groups(result)
#res_grouped2 = get_mean_std_of_groups(res_grouped)
#res_grouped2

In [10]:
def get_trial_overview(res_grouped, plot_title='CP Mean & STD in mV', y1_name='cp_voltage_mean', y1_label='CP Voltage (mV)', y2_name='cp_voltage_std', y2_label='CP Standard Deviation (mV)'):
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=res_grouped.index,
        y=res_grouped[y1_name],
        name=y1_label,
        mode='lines+markers',
        yaxis='y1'
    ))
    
    fig.add_trace(go.Scatter(
        x=res_grouped.index,
        y=res_grouped[y2_name],
        name=y2_label,
        mode='lines+markers',
        yaxis='y2'
    ))
    
    fig.update_layout(
        title=plot_title,
        plot_bgcolor='white',
        xaxis=dict(
            title='Parameter Set',
            showgrid=False,      # no vertikal line
            linecolor='black',   # black axis at bottom
        ),
        yaxis=dict(
            title=y1_label,
            #showgrid=False,      # no horizontal line
            linecolor='blue',
            titlefont=dict(color='blue'),
            tickfont=dict(color='blue'),
        ),
        yaxis2=dict(
            title=y2_label,
            overlaying='y',
            side='right',
            showgrid=False,     # no horizontal line
            linecolor='red',
            titlefont=dict(color='red'),
            tickfont=dict(color='red'),
        ),
        legend=dict(
            x=0.5, y=-0.3,
            xanchor='center',
            orientation='h'
        )
    )
    return fig

def get_trial_std_over_mean(res_grouped, plot_title='CP std vs. CP mean (in mV)', x_name='cp_voltage_mean', x_label='CP Voltage (mV)', y_name='cp_voltage_std', y_label='CP Standard Deviation (mV)'):
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=res_grouped[x_name],   # TODO maybe use mA
        y=res_grouped[y_name],
        mode='markers+text',
        marker=dict(color='green', size=8),
        name='Mean vs Std',
        textposition='top center'
    ))
    
    fig.update_layout(
        title=plot_title,
        xaxis=dict(
            title=x_label,
            showgrid=True,
            gridcolor='lightgrey',
            zeroline=False,
            linecolor='black',
            #range=[0, 2000],
        ),
        yaxis=dict(
            title=y_label, 
            showgrid=True,
            gridcolor='lightgrey',
            zeroline=False,
            linecolor='black',
        ),
        plot_bgcolor='white',
    )
    return fig


In [18]:
def on_analysis_clicked(b):
    global res_grouped
    with analysis_output:
        analysis_output.clear_output()
        print('Evaluating data. This can take some time...')
        res_grouped = get_groups(result.dropna(subset=["sample_id"]))
        res_grouped = get_mean_std_of_groups(res_grouped)
        styled_df = res_grouped.drop(columns=['upload_ids', 'sample_ids']).style.background_gradient(subset=['cp_geom_mean', 'cp_geom_mean_rhe'], cmap='RdYlGn_r')
        fig1 = get_trial_overview(res_grouped, plot_title='CP Mean & STD in mV', y1_name='cp_voltage_mean', y1_label='CP Voltage (mV)', y2_name='cp_voltage_std', y2_label='CP Standard Deviation (mV)')
        fig2 = get_trial_std_over_mean(res_grouped, plot_title='CP std vs. CP mean (in mV)', x_name='cp_voltage_mean', x_label='CP Voltage (mV)', y_name='cp_voltage_std', y_label='CP Standard Deviation (mV)')
        #analysis_output.clear_output()
        display(styled_df)
        fig1.show()
        fig2.show()
    if not res_grouped.empty:
        group_selector.options = list(res_grouped.index)
        with group_select_output:
            group_select_output.clear_output()
            display(group_selector, show_group_details_button)

analysis_button.on_click(on_analysis_clicked)

display(analysis_button, analysis_output)

Button(button_style='info', description='Group and evaluate data', layout=Layout(width='auto'), style=ButtonSt…

Output(outputs=({'name': 'stdout', 'text': 'Evaluating data. This can take some time...\nSkipped measurement "…

### Save Data for Bayesian Optimization

At the moment we use the `cp_geom_mean` column for BO (not the RHE compensated column).

In [12]:
#date_now = datetime.datetime.now()
#file_name = 'baybe_csv/nomad_result_table_maitryi_' + date_now.strftime("%Y%m%d") + '.csv'
#result.to_csv(file_name, index=False)

In [13]:
def save_baybe_table_csv(res_grouped_df, author_name='maitryi'):
    baybe_table = res_grouped_df[BAYBE_COL_NAMES_EXPORT_MAPPING.get(author_selector.value, [''])]
    """
    baybe_table['sweep_speed'] = baybe_table['sweep_speed'].round(0).astype(int)  #TODO what if there is no sweep speed? maybe round this in transformation mappings!1!!!
    #baybe_table = baybe_table.replace('n/a', 0)    # this is from last year where sweep speed could be n/a TODO check if this is also possible this year
    if (baybe_table['potential3'] > 400).any():
        print('Values for potential3 > 400mV will be removed before optimization.')
        baybe_table = baybe_table[baybe_table['potential3'] <= 400]
    if (baybe_table['sweep_speed'] < 10).any():
        print('Values for sweep speed < 10 mV/s will be removed before optimization.')
        baybe_table = baybe_table[baybe_table['sweep_speed'] >= 10]
    """
    baybe_table.to_csv(f'baybe_csv/parameters_bayesian_optimization_{author_name}.csv', index=False, header=True)
    print(f'Saved table for Bayesion Optimization in baybe_csv/parameters_bayesian_optimization_{author_name}.csv')

In [14]:
def on_baybe_clicked(b):
    with baybe_output:
        baybe_output.clear_output()
        first_name = author_selector.value.split()[0].lower()
        save_baybe_table_csv(res_grouped, author_name=first_name)

save_baybe_button.on_click(on_baybe_clicked)

display(save_baybe_button, baybe_output)

Button(button_style='primary', description='Save data for Bayesian Optimization', layout=Layout(width='auto'),…

Output()

### More detailed view on grouped data

In [15]:
def get_oer_cp_compare_plot(time_lists, voltage_lists, labels, plot_title='OER CP Voltage vs Time'):
    fig = go.Figure()

    for time, voltage, label in zip(time_lists, voltage_lists, labels):
        fig.add_trace(go.Scatter(
            x=time,
            y=voltage,
            mode='lines+markers',
            name=label,
            line=dict(width=2),
            marker=dict(size=6)
        ))
    
    fig.update_layout(
        title=plot_title,
        xaxis_title="Time (s)",
        yaxis_title="Voltage (V)",
        template="plotly_white"
    )
    
    fig.show()

def get_cv_compare_plot(dfs):
    fig = go.Figure()

    for i, df in enumerate(dfs, start=1):
        fig.add_trace(go.Scatter(
            x=df['voltage'],
            y=df['current'],
            mode='lines+markers',
            name=df['id'][0],
            line=dict(width=2),
            marker=dict(size=6)
        ))
    
    fig.update_layout(
        title="Activation CV",
        xaxis_title="Voltage (V)",
        yaxis_title="Im (A)",
        template="plotly_white"
    )
    
    fig.show()

In [16]:
def get_group_detail_view(res_grouped, parameter_set_idx):
    group = res_grouped.iloc[parameter_set_idx]

    # ------------- CP --------------
    
    cp_data = get_specific_entrytype_of_upload_ids(url, token, group.upload_ids, 'CE_NOME_Chronopotentiometry',)
    replicates = []
    replicates_time = []
    replicates_id = []
    for measurement in cp_data:
        if cp_pre_step_current_selector.value is not None and measurement.get('properties', {}).get('pre_step_current') != cp_pre_step_current_selector.value / 1000:
            #print(f'Skipped measurement "{measurement.get('name')}", {measurement.get('datetime')}, {measurement.get('samples', [{}])[0].get('name')} because of different pre_step_current.')
            #print(f'({cp_pre_step_current_selector.value / 1000} A != {measurement.get('properties', {}).get('pre_step_current')} A)')
            continue
        replicates.append(measurement.get('voltage'))  # TODO 'voltage_rhe_compensated'
        replicates_time.append(measurement.get('time'))
        replicates_id.append(measurement.get('samples', [''])[0].get('name'))
    mean_all = np.mean(replicates, axis=0)
    std_all = np.std(replicates, axis=0, ddof=1)
    mean_val = mean_all.mean() * 1000 #mV
    std_val = std_all.mean() * 1000 #mV

    get_oer_cp_compare_plot(replicates_time, replicates, replicates_id)
    get_oer_cp_compare_plot([replicates_time[0]], [mean_all], ['V average'], 'OER CP V average')
    get_oer_cp_compare_plot([replicates_time[0]], [std_all], ['V standard deviation'], 'OER CP V standard deviation')

    # ------------- CV --------------

    cv_data = get_specific_entrytype_of_upload_ids(url, token, group.upload_ids, 'CE_NOME_CyclicVoltammetry',)
    first_cycles = []
    last_cycles = []
    for idx, cv in enumerate(cv_data):
        sample_id = cv.get('samples', [''])[0].get('name')
        first_cycle = cv.get('cycles')[1]   # cycle 2 (first complete cycle)
        last_cycle = cv.get('cycles')[-1]
        if len(last_cycle) < len(first_cycle):
            last_cycle = cv.get('cycles')[-2]    # sometimes the last cycle is only half cycle
    
        first_current = first_cycle.get('current')
        first_voltage = np.array(first_cycle.get('voltage_rhe_compensated'))
        first_df = pd.DataFrame({
            'voltage': first_voltage,
            'current': first_current,
            'id': f'first cycle {sample_id}',
        })
        first_cycles.append(first_df)
        
        last_current = last_cycle.get('current')
        last_voltage = np.array(last_cycle.get('voltage_rhe_compensated'))
        last_df = pd.DataFrame({
            'voltage': last_voltage,
            'current': last_current,
            'id': f'last cycle {sample_id}',
        })
        last_cycles.append(last_df)
    get_cv_compare_plot(first_cycles+last_cycles)

In [17]:
def on_group_idx_change(change):
    if change['name'] == 'value' and change['new'] is not None:
        idx = change['new']
        with group_select_output:
            group_select_output.clear_output()
            group_detail_output.clear_output()
            display(group_selector, res_grouped.loc[[idx]].drop(columns=['upload_ids', 'sample_ids']), show_group_details_button)

group_selector.observe(on_group_idx_change, names='value')

def on_show_group_details_button_click(b):
    selected_idx = group_selector.value
    with group_detail_output:
        group_detail_output.clear_output()
        get_group_detail_view(res_grouped, selected_idx)

show_group_details_button.on_click(on_show_group_details_button_click)

try:
    _ = res_grouped  # check if df is already defined
except NameError:
    with group_select_output:
        print('Please run the "Group and evaluate data" button before inspecting individual parameter sets.')

display(group_select_output, group_detail_output)


Output()

Output()